<a href="https://colab.research.google.com/github/mhtrannn/Dynamic-Customer-Segmentation-/blob/main/Step%206_Clustered%20Labeled.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
import hdbscan
from sklearn.metrics import silhouette_score, davies_bouldin_score

# ==========================================
# 1. DATA LOADING & GLOBAL STANDARDIZATION
# ==========================================
df = pd.read_csv("dynamic_profiles_step5_FINAL.csv")
features = ['Recency', 'Frequency', 'Monetary', 'Product_Diversity', 'Purchase_Interval', 'Spending_Trend']
X = df[features].copy()

# Global scaling across all 22 time windows
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Random sampling to optimize Silhouette Score computation
np.random.seed(42)
sample_size = min(20000, len(X_scaled))
sample_indices = np.random.choice(len(X_scaled), size=sample_size, replace=False)
X_sampled = X_scaled[sample_indices]

# ==========================================
# 2. ALGORITHM TUNING & EMPIRICAL COMPARISON
# ==========================================
print("\n--- TABLE 6.1: K-MEANS PARAMETER TUNING ---")
kmeans_results = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_sampled, labels[sample_indices])
    db = davies_bouldin_score(X_scaled, labels)
    kmeans_results.append({'K': k, 'Silhouette': round(sil, 4), 'DBI': round(db, 4)})
print(pd.DataFrame(kmeans_results).to_markdown(index=False))

print("\n--- TABLE 6.2: GMM PARAMETER TUNING ---")
gmm_results = []
for k in range(2, 7):
    gmm = GaussianMixture(n_components=k, random_state=42)
    labels = gmm.fit_predict(X_scaled)
    sil = silhouette_score(X_sampled, labels[sample_indices])
    db = davies_bouldin_score(X_scaled, labels)
    gmm_results.append({'Components (K)': k, 'Silhouette': round(sil, 4), 'DBI': round(db, 4)})
print(pd.DataFrame(gmm_results).to_markdown(index=False))

print("\n--- TABLE 6.3: HDBSCAN PARAMETER TUNING ---")
hdb_results = []
for min_size in [100, 300, 500, 1000]:
    hdb = hdbscan.HDBSCAN(min_cluster_size=min_size)
    labels = hdb.fit_predict(X_scaled)
    noise_pct = round((labels == -1).sum() / len(labels) * 100, 2)
    sil = silhouette_score(X_sampled, labels[sample_indices])
    db = davies_bouldin_score(X_scaled, labels)
    hdb_results.append({'Min Cluster Size': min_size, 'Noise Ratio': f"{noise_pct}%", 'Silhouette': round(sil, 4), 'DBI': round(db, 4)})
print(pd.DataFrame(hdb_results).to_markdown(index=False))

# ==========================================
# 3. FINAL COMPARISON LEADERBOARD
# ==========================================
optimal_kmeans = {'Algorithm': 'K-Means', 'Selected Config': 'K=4', 'Silhouette': 0.3999, 'DBI': 0.8995, 'Noise Ratio': '0.00%'}
optimal_gmm = {'Algorithm': 'Gaussian Mixture Model', 'Selected Config': 'K=4', 'Silhouette': 0.3351, 'DBI': 1.9029, 'Noise Ratio': '0.00%'}
optimal_hdb = {'Algorithm': 'HDBSCAN', 'Selected Config': 'min_size=500', 'Silhouette': 0.1738, 'DBI': 2.1815, 'Noise Ratio': '19.05%'}

print("\n--- TABLE 6.4: FINAL MODEL COMPARISON LEADERBOARD ---")
leaderboard = pd.DataFrame([optimal_kmeans, optimal_gmm, optimal_hdb])
print(leaderboard.to_markdown(index=False))

# ==========================================
# 4. FINAL MODEL FIT & BUSINESS LABELING
# ==========================================
print("\nFitting final optimal model (K-Means K=4) and exporting labeled data...")
final_kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = final_kmeans.fit_predict(X_scaled)

label_mapping = {
    2: 'High-value Loyal',
    0: 'Potential',
    3: 'At-risk / Highly Volatile',
    1: 'Low-value / Inactive'
}

df['Segment'] = df['Cluster'].map(label_mapping)
df.to_csv("dynamic_profiles_step6_CLUSTERED_LABELED.csv", index=False)
print("Export completed successfully: dynamic_profiles_step6_CLUSTERED_LABELED.csv")


--- TABLE 6.1: K-MEANS PARAMETER TUNING ---
|   K |   Silhouette |    DBI |
|----:|-------------:|-------:|
|   2 |       0.3819 | 1.0864 |
|   3 |       0.3976 | 1.0007 |
|   4 |       0.3999 | 0.8995 |
|   5 |       0.4289 | 0.8735 |
|   6 |       0.3997 | 0.8921 |

--- TABLE 6.2: GMM PARAMETER TUNING ---
|   Components (K) |   Silhouette |    DBI |
|-----------------:|-------------:|-------:|
|                2 |       0.3756 | 1.0621 |
|                3 |       0.3848 | 1.5804 |
|                4 |       0.3351 | 1.9029 |
|                5 |       0.2916 | 2.2001 |
|                6 |       0.2914 | 2.0946 |

--- TABLE 6.3: HDBSCAN PARAMETER TUNING ---
|   Min Cluster Size | Noise Ratio   |   Silhouette |    DBI |
|-------------------:|:--------------|-------------:|-------:|
|                100 | 12.03%        |       0.2751 | 1.2571 |
|                300 | 25.27%        |      -0.0268 | 1.8634 |
|                500 | 19.05%        |       0.1738 | 2.1815 |
|              